# Task I — Quantum Computing Part

Two circuits built with Cirq:
1. A 5-qubit circuit with H, CNOT chain, SWAP, and an Rx rotation
2. A swap test circuit comparing two 2-qubit states

## Circuit 1

In [ ]:
import cirq
import math
import numpy as np

In [ ]:
# 5 qubits
qubits = [cirq.NamedQubit(f"q{i}") for i in range(5)]
q0, q1, q2, q3, q4 = qubits

circuit1 = cirq.Circuit()

# (a) Hadamard on every qubit
circuit1.append(cirq.H.on_each(*qubits))

# (b) CNOT chain: (0,1), (1,2), (2,3), (3,4)
circuit1.append([
    cirq.CNOT(q0, q1),
    cirq.CNOT(q1, q2),
    cirq.CNOT(q2, q3),
    cirq.CNOT(q3, q4),
])

# (c) SWAP(0, 4)
circuit1.append(cirq.SWAP(q0, q4))

# (d) Rx(pi/2) on qubit 2
circuit1.append(cirq.rx(math.pi / 2)(q2))

print(circuit1)

In [ ]:
# add measurement and simulate
circuit1.append(cirq.measure(*qubits, key='result'))

sim = cirq.Simulator()
result = sim.run(circuit1, repetitions=100)

print("Measurement results (100 shots):")
print(result.histogram(key='result'))

## Circuit 2 — Swap test

We prepare two 2-qubit states and use the swap test to estimate their overlap:
- State A: $|q_1 q_2\rangle$ where $q_1$ gets a Hadamard and $q_2$ gets $R_x(\pi/3)$
- State B: $|q_3 q_4\rangle$ where both get a Hadamard

Since the states are different, we expect the swap test to output 0 with probability $\frac{1 + |\langle A | B \rangle|^2}{2}$, so not always 0.

In [ ]:
# we need one ancilla per swap test (one for q1 vs q3, one for q2 vs q4)
anc_12 = cirq.NamedQubit("anc_12")
anc_34 = cirq.NamedQubit("anc_34")
q1 = cirq.NamedQubit("q1")
q2 = cirq.NamedQubit("q2")
q3 = cirq.NamedQubit("q3")
q4 = cirq.NamedQubit("q4")

circuit2 = cirq.Circuit()

# -- state preparation --
circuit2.append(cirq.H(q1))              # H on first qubit
circuit2.append(cirq.rx(np.pi / 3)(q2))  # Rx(pi/3) on second qubit
circuit2.append(cirq.H(q3))              # H on third qubit
circuit2.append(cirq.H(q4))              # H on fourth qubit

# -- swap test between q1 and q3 --
circuit2.append(cirq.H(anc_12))
circuit2.append(cirq.CSWAP(anc_12, q1, q3))
circuit2.append(cirq.H(anc_12))
circuit2.append(cirq.measure(anc_12, key='swap_q1_q3'))

# -- swap test between q2 and q4 --
circuit2.append(cirq.H(anc_34))
circuit2.append(cirq.CSWAP(anc_34, q2, q4))
circuit2.append(cirq.H(anc_34))
circuit2.append(cirq.measure(anc_34, key='swap_q2_q4'))

print(circuit2)

In [ ]:
sim = cirq.Simulator()
result2 = sim.run(circuit2, repetitions=1000)

print("Swap test q1 vs q3:", result2.histogram(key='swap_q1_q3'))
print("Swap test q2 vs q4:", result2.histogram(key='swap_q2_q4'))

# q1 and q3 both got H, so they're the same state -> ancilla should almost always be 0
# q2 got Rx(pi/3) and q4 got H, different states -> we expect some 1s

As expected, the swap test between $q_1$ and $q_3$ gives 0 almost every time since both are in the $|+\rangle$ state. The test between $q_2$ and $q_4$ shows a mix of 0s and 1s, reflecting the fact that $R_x(\pi/3)|0\rangle \neq H|0\rangle$.